# Simple RAG with LangChain (Python 3.11.9)

This notebook loads five realistic, fictional training PDFs from the `sample_pdfs` folder and uses them as a RAG knowledge base. Each PDF has three pages of document text.

**Flow:** load PDFs → split text → index chunks → retrieve relevant text → answer questions.

Keep this notebook beside the `sample_pdfs` folder and run cells from top to bottom. You need an OpenAI API key.

## 1. Install packages and set your API key

Run the install cell once. Restart the kernel if prompted. Add your key to a `.env` file beside this notebook:

```text
OPENAI_API_KEY=your-key-here
```

Keep your real key private.

In [ ]:
%pip install -q langchain langchain-classic langchain-community langchain-openai langchain-chroma langchain-text-splitters chromadb pypdf python-dotenv pandas

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Add OPENAI_API_KEY to a .env file or your environment, then rerun this cell.")

CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

## 2. Put the PDF files beside the notebook

Place the provided `sample_pdfs` folder in the same directory as this notebook. It should contain `company_overview.pdf`, `leave_policy.pdf`, `expense_policy.pdf`, `security_policy.pdf`, and `remote_work_policy.pdf`. This notebook reads those files directly; it does not generate them.

## 3. Load all five PDF files

Each page becomes a document, with the PDF filename and page number kept as metadata.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_dir = Path("sample_pdfs")
pdf_files = sorted(pdf_dir.glob("*.pdf"))
if len(pdf_files) != 5:
    raise FileNotFoundError("Expected five PDF files in the sample_pdfs folder. Put that folder beside the notebook.")

pages = []
for pdf_file in pdf_files:
    pages.extend(PyPDFLoader(str(pdf_file)).load())

print("PDF files:", len(pdf_files))
print("Pages loaded:", len(pages))
for pdf_file in pdf_files:
    page_count = sum(1 for page in pages if Path(page.metadata["source"]).name == pdf_file.name)
    print(f"{pdf_file.name}: {page_count} pages")
    assert page_count >= 2, f"{pdf_file.name} should have at least two pages"


## 4. Split documents into chunks

Chunking breaks each PDF into smaller pieces that can be searched.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print("Chunks created:", len(chunks))
print("Example chunk:", chunks[0].page_content[:300])

## 5. Create embeddings and store chunks in Chroma

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="company_documents",
    persist_directory="chroma_db"
)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
print("Vector database is ready.")

## 6. Build the RAG question-answer chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the context below. If the answer is missing, say you could not find it.\n\nContext:\n{context}"),
    ("human", "{input}")
])
answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, answer_chain)

## 7. Ask questions about the PDFs

In [ ]:
def ask(question):
    result = rag_chain.invoke({"input": question})
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Source files:", sorted({Path(doc.metadata["source"]).name for doc in result["context"]}))
    print()
    return result

ask("How many paid annual leave days do full-time employees get?")

In [ ]:
ask("When should employees submit expense claims?")
ask("What should I do if I receive a phishing message?")
ask("How many days per week can an employee work remotely?")

## 8. Validate RAG with five metrics

We will use the same five metrics to compare chunking strategies. Scores range from **0 (poor) to 1 (strong)**.

| Metric | What it checks |
| --- | --- |
| Context precision | Are retrieved chunks relevant to the question? |
| Context recall | Does retrieved context contain the needed information? |
| Faithfulness | Are answer claims supported by retrieved context? |
| Answer relevance | Does the answer address the question? |
| Answer correctness | Does the answer match the reference answer? |

These scores are estimates from an LLM judge. Use them to compare approaches, not as ground truth.

In [ ]:
import json
import re
import pandas as pd

# Ask the judge for JSON, then validate the values with regular Python.
judge = ChatOpenAI(model=CHAT_MODEL, temperature=0)

METRIC_KEYS = [
    "context_precision",
    "context_recall",
    "faithfulness",
    "answer_relevance",
    "answer_correctness",
]

def parse_metric_scores(raw_output):
    """Parse judge JSON and check required fields and score ranges."""
    if hasattr(raw_output, "content"):
        raw_output = raw_output.content
    if isinstance(raw_output, list):
        raw_output = "\n".join(
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in raw_output
        )

    text = str(raw_output).strip()
    # Accept plain JSON or JSON wrapped in a Markdown code fence.
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        text = match.group(1)
    scores = json.loads(text)
    if not isinstance(scores, dict):
        raise ValueError("Judge response must be a JSON object.")

    missing = [key for key in METRIC_KEYS + ["reason"] if key not in scores]
    if missing:
        raise ValueError(f"Judge response is missing fields: {missing}")

    cleaned = {}
    for key in METRIC_KEYS:
        try:
            value = float(scores[key])
        except (TypeError, ValueError):
            raise ValueError(f"Judge field {key!r} must be numeric.")
        if not 0 <= value <= 1:
            raise ValueError(f"Judge field {key!r} must be between 0 and 1; got {value}.")
        cleaned[key] = value
    cleaned["reason"] = str(scores["reason"])
    return cleaned


### Prepare questions and reference answers

The reference answers below come from the five PDFs. The evaluation set is small so it is easy to inspect and keeps API use modest.

In [ ]:
evaluation_set = [
    {
        "question": "How many paid annual leave days do full-time employees receive?",
        "reference": "They receive 18 days of paid annual leave per calendar year.",
    },
    {
        "question": "When should employees submit expense claims?",
        "reference": "Within 30 days of the purchase, with itemized receipts and the required business details.",
    },
    {
        "question": "What should an employee do after receiving a suspected phishing message?",
        "reference": "Do not click links or open attachments; report it to the IT help desk using the approved reporting method.",
    },
    {
        "question": "How many days per week may an employee work remotely under the example policy?",
        "reference": "Up to two days per week, with manager approval.",
    },
    {
        "question": "What approval is needed for a single expense above 10,000 rupees?",
        "reference": "Manager approval is required.",
    },
]

### Run the evaluation

In [ ]:
def score_one(question, reference):
    result = rag_chain.invoke({"input": question})
    context = "\n\n".join(doc.page_content for doc in result["context"])

    judge_prompt = f"""You are a careful evaluator of a retrieval-augmented QA system.
Return only a valid JSON object with these exact keys: context_precision, context_recall, faithfulness, answer_relevance, answer_correctness, reason.
The five metric values must be numbers from 0 to 1. The reason must be one short explanation. Do not include Markdown or extra text.

Score each metric from 0 to 1. Use 1 for fully successful, 0 for unsuccessful, and intermediate values for partial success.

Context precision: relevance of retrieved context to the question.
Context recall: whether retrieved context contains the information needed by the reference answer.
Faithfulness: whether each factual claim in the generated answer is supported by the context.
Answer relevance: whether the answer directly addresses the question.
Answer correctness: whether the answer is substantively consistent with the reference answer.

Question: {question}
Reference answer: {reference}
Retrieved context: {context}
Generated answer: {result['answer']}
"""
    scores = parse_metric_scores(judge.invoke(judge_prompt))
    return {
        "Question": question,
        "Reference answer": reference,
        "Generated answer": result["answer"],
        "Context precision": scores["context_precision"],
        "Context recall": scores["context_recall"],
        "Faithfulness": scores["faithfulness"],
        "Answer relevance": scores["answer_relevance"],
        "Answer correctness": scores["answer_correctness"],
        "Judge note": scores["reason"],
    }

rows = [score_one(item["question"], item["reference"]) for item in evaluation_set]
results_df = pd.DataFrame(rows)
display(results_df)

### Average metric scores

In [ ]:
metric_columns = [
    "Context precision", "Context recall", "Faithfulness",
    "Answer relevance", "Answer correctness"
]
average_scores = results_df[metric_columns].mean().round(2)
display(average_scores.to_frame("Average score (0–1)"))

## 10. Compare chunking strategies

A chunk is a small piece of a document. Smaller chunks can make search more focused, while larger chunks keep more surrounding detail. Overlap repeats a little text between neighboring chunks so ideas are less likely to be split apart.

We will compare three simple settings using the same questions and metrics. This takes additional API calls because each question is answered and judged once per strategy.

In [ ]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_settings = {
    "Small (250 / 25)": (250, 25),
    "Medium (500 / 50)": (500, 50),
    "Large (1000 / 100)": (1000, 100),
}

comparison_rows = []
original_rag_chain = rag_chain

for strategy, (chunk_size, chunk_overlap) in chunk_settings.items():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    strategy_chunks = splitter.split_documents(pages)
    strategy_db = Chroma.from_documents(
        documents=strategy_chunks,
        embedding=embeddings,
        collection_name="chunks_" + str(chunk_size),
    )
    strategy_retriever = strategy_db.as_retriever(search_kwargs={"k": 3})
    rag_chain = create_retrieval_chain(strategy_retriever, answer_chain)

    strategy_rows = [
        score_one(item["question"], item["reference"])
        for item in evaluation_set
    ]
    strategy_df = pd.DataFrame(strategy_rows)
    comparison_rows.append({
        "Strategy": strategy,
        "Chunks created": len(strategy_chunks),
        **strategy_df[metric_columns].mean().to_dict(),
    })

rag_chain = original_rag_chain
comparison_df = pd.DataFrame(comparison_rows).set_index("Strategy")
display(comparison_df.round(2))

### How to read the comparison

Compare the five average scores across the rows. A higher score means that strategy did better on this small example set. The number of chunks shows how the settings change the size of the searchable collection.

Chunking matters because it changes what information can be retrieved together. If chunks are too small, useful context may be split up. If they are too large, search may return extra unrelated text. Overlap can help preserve details at chunk boundaries. The best setting depends on your documents and questions, so try a few and compare the results.

### Reading the results

A low **context precision** score suggests retrieval is bringing in irrelevant chunks. Low **context recall** suggests useful evidence was missed. Low **faithfulness** suggests the answer includes unsupported claims. Low **answer relevance** or **correctness** suggests the response needs improvement. Use the judge notes and source text to investigate low scores.

## Summary

The notebook loads five PDF files, builds a retrieval-based question-answer chain, and evaluates it with five judge-based metrics: context precision, context recall, faithfulness, answer relevance, and answer correctness.
